In [8]:
import json
from pydoc import stripid


def check_property_string(json_file, property_name):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    counter = 0
    for obj in data:
        if property_name not in obj or not isinstance(obj[property_name], int):
            counter+=1
            print(f"Invalid object with id: {obj['~id']} and name: {obj['Company']}")

    print(f'Total object count: {len(data)}', f'Invalid object count: {counter}', sep='\n')

In [6]:
check_property_string('../data/raw/25ipo-listing-date-check-status-price-bse-nse.json', 'NSE Symbol')

Invalid object with id: 149 and name: Precision Pipes & Profiles Co.Ltd.
Invalid object with id: 149 and name: Precision Pipes & Profiles Co.Ltd.
Invalid object with id: 134 and name: Barak Valley Cements Ltd.
Invalid object with id: 134 and name: Barak Valley Cements Ltd.
Invalid object with id: 846 and name: Lemon Tree Hotels Ltd.
Invalid object with id: 839 and name: ICICI Securities Ltd.
Invalid object with id: 835 and name: Mishra Dhatu Nigam Ltd.
Invalid object with id: 834 and name: Sandhar Technologies Ltd.
Invalid object with id: 832 and name: Karda Constructions Ltd.
Invalid object with id: 831 and name: Hindustan Aeronautics Ltd.
Invalid object with id: 822 and name: Bandhan Bank Ltd.
Invalid object with id: 827 and name: Bharat Dynamics Ltd.
Invalid object with id: 816 and name: H.G.Infra Engineering Ltd.
Invalid object with id: 810 and name: Aster DM Healthcare Ltd.
Invalid object with id: 798 and name: Galaxy Surfactants Ltd.
Invalid object with id: 792 and name: Amber En

In [9]:
check_property_string('../data/raw/25ipo-listing-date-check-status-price-bse-nse.json', "BSE Scrip Code")

Invalid object with id: 149 and name: Precision Pipes & Profiles Co.Ltd.
Invalid object with id: 149 and name: Precision Pipes & Profiles Co.Ltd.
Invalid object with id: 134 and name: Barak Valley Cements Ltd.
Invalid object with id: 134 and name: Barak Valley Cements Ltd.
Invalid object with id: 620 and name: Central Depository Services (India) Ltd.
Invalid object with id: 611 and name: BSE Ltd.
Total object count: 804
Invalid object count: 6


In [17]:
from bs4 import BeautifulSoup
import re
from datetime import datetime

def parse_price_band(text):
    """
    Extract numeric low/high price.
    If empty or invalid → return (None, None)
    """

    if not text or text.strip() == "":
        return None, None

    nums = re.findall(r"\d+\.?\d*", text)

    if not nums:
        return None, None

    nums = [float(n) for n in nums]

    if len(nums) == 1:
        return nums[0], nums[0]

    return min(nums), max(nums)

def normalize_header(text):
    text = text.strip().lower()
    text = re.sub(r"\(.*?\)", "", text)  # remove anything in ()
    text = text.strip()
    return text

def parse_date(text):
    """
    Convert various date formats to ISO (YYYY-MM-DD).
    Return None if invalid or empty.
    """

    if not text or text.strip() == "":
        return None

    text = text.strip()

    formats = [
        "%d-%m-%Y",
        "%d/%m/%Y",
        "%d %b %Y",
        "%d %B %Y"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(text, fmt).date().isoformat()
        except ValueError:
            pass

    return None
def parse_html_table(file_path):

    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "lxml")

    table = soup.find("table", {"id": "ctl00_ContentPlaceHolder1_tblID"})

    if table is None:
        raise ValueError("Required table not found — invalid HTML")

    headers = [
        normalize_header(th.get_text(separator=" ", strip=True))
        for th in table.select("thead th")
    ]

    required_headers = [
        "security name",
        "exchange platform",
        "start date",
        "end date",
        "offer price"
    ]

    for h in required_headers:
        if not any(h in header for header in headers):
            raise ValueError(f"Missing required column: {h}")

    rows = table.select("tbody tr")

    results = []

    for row in rows:
        print(f'Parsing row: {row}')
        cols = [td.get_text(strip=True) for td in row.find_all("td")]

        if len(cols) != len(headers):
            raise ValueError("Row column count mismatch — malformed HTML")

        row_data = dict(zip(headers, cols))

        platform = row_data["exchange platform"].strip()

        if platform not in ["SME", "MainBoard"]:
            raise ValueError(f"Unexpected Exchange Platform: {platform}")

        if platform != "MainBoard":
            continue

        price_low, price_high = parse_price_band(row_data["offer price"])

        obj = {
            "security_name": row_data["security name"],
            "start_date": parse_date(row_data["start date"]),
            "end_date": parse_date(row_data["end date"]),
            "price_band_low": price_low,
            "price_band_high": price_high
        }

        results.append(obj)

    return results

In [18]:
import json
data = parse_html_table("data/bseTable.txt")

print(json.dumps(data, indent=2))

Parsing row: <tr>
<td class="TTRow_left">
<a class="tablebluelink" href="/markets/publicIssues/DisplayIPO.aspx?id=">ADMACH SYSTEMS LIMITED</a>
</td>
<td class="TTRow">SME</td>
<td class="TTRow">23-12-2025</td>
<td class="TTRow">26-12-2025</td>
<td class="TTRow_right">227.00 - 239.00</td>
<td class="TTRow_right">10.00</td>
<td class="TTRow">Book Building</td>
</tr>
Parsing row: <tr>
<td class="TTRow_left">
<a class="tablebluelink" href="/markets/publicIssues/DisplayIPO.aspx?id=">APOLLO TECHNO INDUSTRIES LIMITED</a>
</td>
<td class="TTRow">SME</td>
<td class="TTRow">23-12-2025</td>
<td class="TTRow">26-12-2025</td>
<td class="TTRow_right">123.00 - 130.00</td>
<td class="TTRow_right">10.00</td>
<td class="TTRow">Book Building</td>
</tr>
Parsing row: <tr>
<td class="TTRow_left">
<a class="tablebluelink" href="/markets/publicIssues/DisplayIPO.aspx?id=">BAI-KAKAJI POLYMERS LIMITED</a>
</td>
<td class="TTRow">SME</td>
<td class="TTRow">23-12-2025</td>
<td class="TTRow">26-12-2025</td>
<td cla

In [23]:
data[1]

{'security_name': 'KSH International Limited',
 'start_date': '2025-12-16',
 'end_date': '2025-12-18',
 'price_band_low': 365.0,
 'price_band_high': 384.0}

In [24]:
for ipo in data:
    print(ipo['security_name'])

Gujarat Kidney and Super Speciality Limited
KSH International Limited
ICICI PRUDENTIAL ASSET MANAGEMENT COMPANY LIMITED
Nephrocare Health Services Limited
Park Medi World Limited
CORONA Remedies Limited
Wakefit Innovations Limited
Aequs Limited
MEESHO LIMITED
Vidya Wires Limited
Sudeep Pharma Limited
EXCELSOFT TECHNOLOGIES LIMITED
Capillary Technologies India Limited
Fujiyama Power Systems Limited
TENNECO CLEAN AIR INDIA LIMITED
EMMVEE Photovoltaic Power Limited
Physicswallah Limited
Pine Labs Limited
Billionbrains Garage Ventures Limited
Lenskart Solutions Limited
Studds Accessories Limited
Orkla India Limited
MIDWEST LIMITED
Canara HSBC Life Insurance Company Limited
Canara Robeco Asset Management Company Limited
Rubicon Research Limited
LG Electronics India Limited
Tata Capital Limited
WeWork India Management Limited
Advance Agrolife Limited
OM FREIGHT FORWARDERS LIMITED
FABTECH TECHNOLOGIES LIMITED
GLOTTIS LIMITED
PACE DIGITEK LIMITED
Jinkushal Industries Limited
TruAlt Bioenergy L

In [28]:
import json
import re
from rapidfuzz import process, fuzz


# -------------------------------
# Normalization function
# -------------------------------

def normalize_company(name):

    if not name:
        return ""

    name = name.lower()

    name = name.replace("&", " and ")

    name = re.sub(
        r"\b(limited|ltd|ltd\.|pvt|private|inc|corp|corporation|technologies|technology|industries|financial|services|solutions|holdings)\b",
        "",
        name
    )

    name = re.sub(r"[^a-z0-9]", "", name)

    return name.strip()


# -------------------------------
# Build lookup dictionary
# -------------------------------

lookup = {}

for row in data:
    key = normalize_company(row["security_name"])
    lookup[key] = row

lookup_keys = list(lookup.keys())


# -------------------------------
# Load second dataset
# -------------------------------

with open("../data/filtered/nse/nseAggregatedData.json", "r") as f:
    target_data = json.load(f)


# -------------------------------
# Counters
# -------------------------------

total_ipos = len(target_data)
intact_ipos = 0
exact_matches = 0
fuzzy_matches = 0


# -------------------------------
# Matching and filling
# -------------------------------

for obj in target_data:

    # skip if already populated
    if all(obj.get(k) is not None for k in [
        "ipoStartDate",
        "ipoEndDate",
        "price_band_low",
        "price_band_high"
    ]):
        intact_ipos += 1
        continue

    normalized = normalize_company(obj["Company"])

    match = None

    # exact match
    if normalized in lookup:
        match = lookup[normalized]
        exact_matches += 1

    else:

        best = process.extractOne(
            normalized,
            lookup_keys,
            scorer=fuzz.token_sort_ratio
        )

        if best and best[1] >= 90:

            matched_key = best[0]
            score = best[1]

            match = lookup[matched_key]

            fuzzy_matches += 1

            print(
                f"FUZZY MATCH ({score}): "
                f"{obj['Company']}  --->  {match['security_name']}"
            )

    if match:

        obj["ipoStartDate"] = match.get("start_date")
        obj["ipoEndDate"] = match.get("end_date")
        obj["price_band_low"] = match.get("price_band_low")
        obj["price_band_high"] = match.get("price_band_high")


# -------------------------------
# Save updated dataset
# -------------------------------

# with open("updated_target_data.json", "w") as f:
#     json.dump(target_data, f, indent=2)


# -------------------------------
# Summary
# -------------------------------

print("------ IPO Matching Summary ------")
print(f"Total IPOs in dataset: {total_ipos}")
print(f"IPOs left intact: {intact_ipos}")
print(f"IPOs matched with exact match: {exact_matches}")
print(f"IPOs matched with fuzzy match: {fuzzy_matches}")
print(f"Total processed (exact + fuzzy): {exact_matches + fuzzy_matches}")
print("----------------------------------")

FUZZY MATCH (90.9090909090909): Canara HSBC Life Insurance Co.Ltd.  --->  Canara HSBC Life Insurance Company Limited
FUZZY MATCH (92.95774647887323): Shree Tirupati Balajee Agro Trading Co.Ltd.  --->  Shree Tirupati Balajee Agro Trading Company Limited
FUZZY MATCH (90.9090909090909): Hariom Pipe Industries Ltd.  --->  HARIOM PIPE INDUSTRIES LT
FUZZY MATCH (91.52542372881356): Precision Pipes & Profiles Co.Ltd.  --->  Precision Pipes and Profiles Company Ltd
FUZZY MATCH (91.52542372881356): HDFC Standard Life Insurance Co.Ltd.  --->  HDFC STANDARD LIFE INSURANCE COMPANY LTD
FUZZY MATCH (92.3076923076923): ICICI Lombard General Insurance Co.Ltd.  --->  ICICI LOMBARD GENERAL INSURANCE COMPANY LTD
FUZZY MATCH (92.3076923076923): ICICI Prudential Life Insurance Co.Ltd.  --->  ICICI PRUDENTIAL LIFE INSURANCE COMPANY LIMITED
FUZZY MATCH (97.14285714285714): Shree Pushkar Chemicals & Fertilisers Ltd.  --->  SHREE PUSHKAR CHEMICALS AND FERTILIZERS LIMITED
FUZZY MATCH (97.77777777777777): Niraj 

In [29]:
with open("../data/filtered/bseNseAggregatedData.json", "w") as f:
    json.dump(target_data, f, indent=2)

In [31]:
import json

# load dataset.json
with open("../data/nseAggregated/dataset.json", "r") as f:
    dataset = json.load(f)

# build lookup by id
dataset_lookup = {obj["~id"]: obj for obj in dataset}

updates = 0
missing_ids = 0


for obj in target_data:

    ipo_id = obj["~id"]

    if ipo_id not in dataset_lookup:
        missing_ids += 1
        continue

    source = dataset_lookup[ipo_id]

    # update fields
    source["ipoStartDate"] = obj.get("ipoStartDate")
    source["ipoEndDate"] = obj.get("ipoEndDate")
    source["price_band_low"] = obj.get("price_band_low")
    source["price_band_high"] = obj.get("price_band_high")

    updates += 1

print(f'DAtaset size: {len(dataset)}', f'target_data size: {len(target_data)}', sep='\n')
print("Update Summary")
print("--------------")
print("records updated:", updates)
print("ids not found:", missing_ids)

DAtaset size: 720
target_data size: 802
Update Summary
--------------
records updated: 719
ids not found: 83


In [32]:
import csv
final_data = list(dataset_lookup.values())
with open("../data/aggregated/dataset.json", "w") as f:
    json.dump(final_data, f, indent=2)


# -----------------------------
# export CSV
# -----------------------------

if final_data:

    fieldnames = final_data[0].keys()

    with open("../data/aggregated/dataset.csv", "w", newline="") as f:

        writer = csv.DictWriter(f, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(final_data)